# All sessions — the random timeout / banishment task

Performance **across sessions and mice**. The unit of analysis is a **session** (and, aggregated, an
**animal**) — never the individual trial. Trials appear once as a small count summary; every comparison
after that is one dot per session (colour = mouse) with the animal mean marked.

**Caching:** the tables are saved after they are built, so you only pay the slow log-reading once. Set
`REBUILD = False` (the default) to **load the saved tables** and skip re-reading every log; set
`REBUILD = True` when the data or the code changed.

1. load or build the tables  2. inventory  3. counts (the only trial view)  4. choice & avoidance
5. timing/distance/path by effect  6. occupancy (per session)  7. heading error (per session)  8. export

In [ ]:
import sys, json, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore', message='.*All-NaN slice.*')

_HERE = Path.cwd()
_MP = None
for _b in (_HERE, *_HERE.parents):
    if (_b / 'mixed_perf.py').exists(): _MP = _b; break
    if (_b / 'mixed_protocol' / 'mixed_perf.py').exists(): _MP = _b / 'mixed_protocol'; break
assert _MP is not None, f'cannot locate mixed_perf.py from {_HERE}'
sys.path.insert(0, str(_MP))
import mixed_perf as mp
try:                       # keep figures in memory so save_report() at the end can collect them all
    get_ipython().run_line_magic('config', 'InlineBackend.close_figures = False')
except Exception: pass
print('mixed_perf loaded from', _MP)

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
# Expected layout on the server:   MAIN_DIR / <mouse> / <session> / log.json
MAIN_DIR = '/path/to/MAIN_DIR'      # <-- root folder that holds the MOUSE folders
VIEW_SCALE = mp.DEFAULT_VIEW_SCALE   # world zoom (not in the log; 0.35 default)
PATTERN = '*/*/log.json'             # <mouse>/<session>/log.json ; falls back to a recursive search
REBUILD = False                      # False = load saved tables if present ; True = re-read every log

MAIN_DIR = Path(MAIN_DIR)
assert MAIN_DIR.exists(), f'MAIN_DIR does not exist: {MAIN_DIR}'
CACHE = MAIN_DIR / 'mixed_protocol_df'

## 1 — load or build the tables

If a saved cache exists and `REBUILD` is False, the tables load in a second (no logs read). Otherwise
every log is read once, and **three tables** are built and saved together with the two precomputed
inputs the map/heading sections need (`occupancy.pkl`, `heading.pkl`) — so a later load reproduces the
whole notebook without touching the logs:

- **ALL** — one row per collection · **SUM** — one row per **session** · **PS** — one row per **session×effect**.

In [ ]:
_files = ['all.pkl', 'summary.pkl', 'per_effect.pkl', 'occupancy.pkl', 'heading.pkl']
_have = CACHE.exists() and all((CACHE / f).exists() for f in _files)

if not REBUILD and _have:
    ALL = pd.read_pickle(CACHE / 'all.pkl'); SUM = pd.read_pickle(CACHE / 'summary.pkl')
    PS = pd.read_pickle(CACHE / 'per_effect.pkl')
    OCC = pickle.load(open(CACHE / 'occupancy.pkl', 'rb')); HEAD = pickle.load(open(CACHE / 'heading.pkl', 'rb'))
    print(f'LOADED cached tables from {CACHE}  (set REBUILD=True to rebuild)')
else:
    found = mp.find_sessions(MAIN_DIR, PATTERN)
    SESSIONS = []
    for p, log in mp._progress(found, 'building session tables'):
        mouse, session, mouse_folder = mp.session_label(p, log, main_dir=MAIN_DIR)
        df = mp.build_session_df(log, view_scale=VIEW_SCALE, session=session, mouse=mouse)
        df['mouse_folder'] = mouse_folder
        df['date'] = str(log.get('experiment_data', {}).get('datetime', ''))[:19]
        SESSIONS.append((mouse, session, log, df))
    assert len(SESSIONS), 'no mixed-protocol sessions found under MAIN_DIR (check the path / PATTERN / layout)'

    ALL = pd.concat([df for _, _, _, df in SESSIONS], ignore_index=True)
    ALL['head_deg'] = np.degrees(np.arccos(ALL['heading_align'].clip(-1, 1)))
    SUM = pd.DataFrame([{**mp.session_summary(log, df),
                         'date': df['date'].iloc[0], 'mouse_folder': df['mouse_folder'].iloc[0]}
                        for _, _, log, df in SESSIONS])
    PS = (ALL.groupby(['mouse', 'session', 'effect'])
             .agg(n=('idx', 'size'), time_s=('dt_prev_ms', lambda v: float(np.nanmean(v)) / 1000),
                  dist=('dist_prev', 'mean'), pe=('path_efficiency', 'median'), head_deg=('head_deg', 'median'))
             .reset_index())

    HALF, BINS = 900, 45; GRID = np.linspace(-8, 0, 60)     # precompute the map + heading inputs, ONE pass
    OCC = {'half': HALF}; HEAD = {'grid': GRID}
    _acc = {e: [np.zeros((BINS, BINS)), 0] for e, _ in mp.TYPES}; _hd = {e: [] for e, _ in mp.TYPES}
    for _, _, log, df in mp._progress(SESSIONS, 'occupancy + heading'):
        for e, _ in mp.TYPES:
            ox, oy, dt = mp.collection_offsets(log, df, e, window_s=3.0)
            if ox.size >= 5:
                Hh, _, _ = np.histogram2d(ox, oy, bins=BINS, range=[[-HALF, HALF], [-HALF, HALF]], weights=dt)
                if Hh.sum() > 0: _acc[e][0] += Hh / Hh.sum(); _acc[e][1] += 1
            curves = mp.collected_curves_time(log, df, e, window_s=8.0)
            if curves:
                M = np.vstack([np.interp(GRID, t, er, left=np.nan, right=np.nan) for t, er in curves])
                _hd[e].append(np.nanmedian(M, axis=0))
    for e, _ in mp.TYPES:
        acc, ns = _acc[e]; OCC[e] = dict(map=(acc / ns if ns else acc), n=ns); HEAD[e] = _hd[e]

    CACHE.mkdir(exist_ok=True)
    ALL.to_pickle(CACHE / 'all.pkl'); SUM.to_pickle(CACHE / 'summary.pkl'); PS.to_pickle(CACHE / 'per_effect.pkl')
    pickle.dump(OCC, open(CACHE / 'occupancy.pkl', 'wb')); pickle.dump(HEAD, open(CACHE / 'heading.pkl', 'wb'))
    print(f'BUILT and SAVED tables -> {CACHE}')

print(f'{SUM.shape[0]} sessions | {SUM.mouse.nunique()} mice | ALL {ALL.shape} | SUM {SUM.shape} | PS {PS.shape}')

## 2 — inventory + shared style

One row per session (from `SUM`); the colour of each mouse is fixed here and reused by every plot.

In [ ]:
inv = SUM[['mouse', 'mouse_folder', 'session', 'date', 'n_coll',
           'n_reward', 'n_timeout', 'n_banish', 'n_escape']].sort_values(['mouse', 'date']).reset_index(drop=True)

mice = sorted(SUM['mouse'].unique())
_cm = plt.cm.tab10(np.linspace(0, 1, 10)); MCOL = {m: _cm[i % 10] for i, m in enumerate(mice)}
from matplotlib.lines import Line2D
_mlegend = [Line2D([0], [0], marker='o', ls='', color=MCOL[m], mec='k', label=m) for m in mice]

def dots_by_mouse(ax, data, key, ylim=None):
    '''one dot per SESSION (colour = mouse) at that mouse's x, plus a diamond = the ANIMAL mean.'''
    rng = np.random.default_rng(0)
    for i, m in enumerate(mice):
        v = data[data.mouse == m][key].dropna().values
        ax.scatter(np.full(len(v), i) + rng.uniform(-.11, .11, len(v)), v, s=42,
                   color=MCOL[m], edgecolor='k', lw=.4, alpha=.8, zorder=2)
        if len(v): ax.scatter([i], [np.nanmean(v)], marker='D', s=95, color=MCOL[m], edgecolor='k', lw=1.3, zorder=3)
    ax.set_xticks(range(len(mice))); ax.set_xticklabels(mice, rotation=30, fontsize=7)
    if ylim: ax.set_ylim(*ylim)

def dots_by_effect(ax, ps, key, ylim=None):
    '''one dot per SESSION (colour = mouse) grouped by effect, plus a bar = the across-session mean.'''
    effs = ['single_reward', 'banish', 'timeout']; rng = np.random.default_rng(0)
    for i, e in enumerate(effs):
        sub = ps[ps.effect == e]
        for m in mice:
            v = sub[sub.mouse == m][key].dropna().values
            ax.scatter(np.full(len(v), i) + rng.uniform(-.12, .12, len(v)), v, s=36,
                       color=MCOL[m], edgecolor='k', lw=.3, alpha=.8, zorder=2)
        allv = sub[key].dropna().values
        if len(allv): ax.scatter([i], [np.nanmean(allv)], marker='_', s=650, color='k', zorder=4)
    ax.set_xticks(range(len(effs))); ax.set_xticklabels([mp.ELABEL[e] for e in effs])
    if ylim: ax.set_ylim(*ylim)

print('dots = sessions (colour = mouse), diamond = animal mean')
inv

## 3 — dataset at a glance

The only trial-level view: sessions & mice, sessions per mouse, total collections by effect, and — of
the rewards — how many landed at each multiplier (combo) level.

In [ ]:
n_sess, n_mice = SUM.shape[0], len(mice)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
spm = inv.groupby('mouse').size().reindex(mice)
ax[0].bar(range(len(mice)), spm.values, color=[MCOL[m] for m in mice])
for i, v in enumerate(spm.values): ax[0].text(i, v + 0.05, str(int(v)), ha='center')
ax[0].set_xticks(range(len(mice))); ax[0].set_xticklabels(mice, rotation=30, fontsize=7)
ax[0].set_ylabel('sessions'); ax[0].set_title(f'{n_sess} sessions across {n_mice} mice')

effs = ['single_reward', 'timeout', 'banish', 'unbanish']
tot = [int((ALL.effect == e).sum()) for e in effs]
ax[1].bar(range(4), tot, color=[mp.COLR[e] for e in effs])
for i, v in enumerate(tot): ax[1].text(i, v + max(tot)*0.01, str(v), ha='center', fontsize=9)
ax[1].set_xticks(range(4)); ax[1].set_xticklabels([mp.ELABEL[e] for e in effs])
ax[1].set_ylabel('collections (all sessions)'); ax[1].set_title('total collections by effect')

mult = ALL.loc[ALL.valence == 'positive', 'multiplier'].dropna().astype(int)
mc = mult.value_counts().sort_index()
ax[2].bar(mc.index, mc.values, color=mp.COLR['single_reward'])
for x, v in zip(mc.index, mc.values): ax[2].text(x, v + max(mc.values)*0.01, str(v), ha='center', fontsize=9)
ax[2].set_xlabel('reward multiplier (combo level)'); ax[2].set_ylabel('reward collections')
ax[2].set_title(f'rewards by multiplier level ({len(mult)} rewards)')
plt.tight_layout(); plt.show()

## 4 — choice & avoidance: session & animal as a data point

Each dot is **one session** (colour = mouse); the **diamond is the animal mean**. Stochastic p per
session (small = above chance / hazard avoided; red line p = 0.05), then reward rate, win-stay, bonus
captured, mean multiplier, collections per session.

In [ ]:
panels = [('p_all', 'stochastic p — all negatives', (0, 1.02)), ('p_timeout', 'stochastic p — timeout', (0, 1.02)),
          ('p_banish', 'stochastic p — banishment', (0, 1.02)), ('reward_rate', 'reward rate', (0, 1.02)),
          ('win_stay', 'win-stay  P(reward|reward)', (0, 1.02)), ('bonus_captured', 'multiplier bonus captured', (0, 1.02)),
          ('mult_mean', 'mean multiplier', None), ('n_coll', 'collections per session', None)]
fig, axes = plt.subplots(2, 4, figsize=(17, 8)); axes = axes.ravel()
for a, (key, lbl, ylim) in zip(axes, panels):
    dots_by_mouse(a, SUM, key, ylim=ylim); a.set_title(lbl, fontsize=10)
    if key.startswith('p_'): a.axhline(0.05, ls='--', color='r', lw=.9)
axes[0].legend(handles=_mlegend, fontsize=7, title='mouse', loc='upper left')
plt.suptitle('choice & avoidance — dot = session (colour = mouse), diamond = animal mean', y=1.01)
plt.tight_layout(); plt.show()

## 5 — timing, distance & path, by effect (session as a data point)

Per session, the average **time** and **distance** to collect a reward vs a timeout vs a banishment,
the **path efficiency** of those approaches (1 = beeline), and **how many** of each per session. Each dot
is one session (colour = mouse); the black bar is the across-session mean.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 4.3))
dots_by_effect(ax[0], PS, 'time_s'); ax[0].set_ylabel('seconds'); ax[0].set_title('avg time to collect (per session)')
dots_by_effect(ax[1], PS, 'dist'); ax[1].set_ylabel('world units'); ax[1].set_title('avg distance to collect')
dots_by_effect(ax[2], PS, 'pe', ylim=(0, 1.02)); ax[2].set_ylabel('path efficiency'); ax[2].set_title('path efficiency (1 = beeline)')
dots_by_effect(ax[3], PS, 'n'); ax[3].set_ylabel('collections'); ax[3].set_title('collections per session')
ax[0].legend(handles=_mlegend, fontsize=7, title='mouse', loc='upper right')
plt.suptitle('per-effect, per session (dot = session, colour = mouse; bar = across-session mean)')
plt.tight_layout(); plt.show()

## 6 — occupancy around each icon, averaged per session

Icon-centred avatar occupancy (±3 s around each collection), **normalised within each session and
averaged across sessions** (precomputed in section 1). Colour = mean fraction of the near-icon time.
Star = the icon.

In [ ]:
HALF = OCC['half']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for a, (e, lbl) in zip(axes, mp.TYPES):
    M, ns = OCC[e]['map'], OCC[e]['n']
    im = a.imshow(M.T, origin='lower', extent=[-HALF, HALF, -HALF, HALF], cmap='magma', aspect='equal')
    fig.colorbar(im, ax=a, fraction=0.046, pad=0.04, label='mean fraction of near-icon time')
    a.plot(0, 0, marker='*', ms=15, color='cyan', mec='k')
    a.set_title(f'{lbl}  (avg of {ns} sessions)'); a.set_xlabel('x - icon (wu)'); a.set_ylabel('y - icon (wu)')
plt.suptitle('occupancy around the collected icon — averaged per session (±3 s)')
plt.tight_layout(); plt.show()

## 7 — heading error toward the collected icon, per session

Heading error in **degrees** (0 = facing the icon, 180 = away) over the last 8 s before collection. Each
**thin line is one session's median**, the **bold line is the mean across sessions** (precomputed in
section 1). One panel per effect.

In [ ]:
GRID = HEAD['grid']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
for a, (e, lbl) in zip(axes, mp.TYPES):
    curves = HEAD[e]
    for c in curves:
        a.plot(GRID, c, color=mp.COLR[e], alpha=0.28, lw=1)
    if curves:
        M = np.vstack(curves); ok = np.isfinite(M).sum(0) > 0
        mean = np.full(M.shape[1], np.nan); mean[ok] = np.nanmean(M[:, ok], axis=0)
        a.plot(GRID, mean, color=mp.COLR[e], lw=3, label='mean of sessions')
    a.axhline(90, ls=':', color='0.6'); a.set_ylim(0, 180)
    a.set_xlabel('time to collection (s)'); a.set_title(f'{lbl}  ({len(curves)} sessions)'); a.legend(fontsize=8)
axes[0].set_ylabel('heading error (deg)\n0 = facing icon · 180 = away')
plt.suptitle('heading error toward the collected icon — thin = per-session median, bold = mean of sessions')
plt.tight_layout(); plt.show()

## — export: save the whole report as a PDF or PNGs

Run **after `Run All`**. `save_report('pdf')` → one multi-page PDF; `save_report('png')` → individual
PNGs. Pass `path=` to redirect.

In [ ]:
_stem = 'mixed_protocol'
def save_report(fmt='pdf', path=None, dpi=130):
    '''Save every figure produced above (in creation order) as ONE PDF, or as individual PNGs.'''
    from matplotlib.backends.backend_pdf import PdfPages
    figs = [plt.figure(n) for n in plt.get_fignums()]
    if not figs:
        print('no open figures -- do "Run All" first, then run this cell'); return
    if fmt == 'pdf':
        p = Path(path) if path else MAIN_DIR / 'mixed_protocol_performance.pdf'
        p.parent.mkdir(parents=True, exist_ok=True)
        with PdfPages(p) as pdf:
            for f in figs: pdf.savefig(f, bbox_inches='tight')
        print(f'saved {len(figs)}-page PDF -> {p}')
    else:
        d = Path(path) if path else CACHE / 'report_png'
        d.mkdir(parents=True, exist_ok=True)
        for i, f in enumerate(figs, 1):
            f.savefig(d / f'{_stem}_fig{i:02d}.png', dpi=dpi, bbox_inches='tight')
        print(f'saved {len(figs)} PNGs -> {d}')

# save_report('pdf')      # <- uncomment to write the PDF
# save_report('png')      # <- or the PNGs
print('report export ready: call  save_report("pdf")  or  save_report("png")')